# DS4/GSE278450 — post-QC Scrublet doublet diagnostic

This notebook reproduces the original Scrublet check performed on the
complete post-QC DS4 object.

The historical run predicted:

- 310 doublets;
- 87,784 singlets.

Predicted doublets were retained as diagnostic metadata and were not
removed from the main analysis. The DS4 analytical cell count therefore
remains 88,094.

Input:

`data/processed/DS4_GSE278450/GSE278450_postQC.h5ad`

Output:

`data/processed/DS4_GSE278450/GSE278450_postQC_scrublet_checked.h5ad`

In [ ]:
import importlib.util
import inspect
import json
import os
import sys
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp

print("Python:", sys.version)
print("scanpy:", sc.__version__)

In [ ]:
def find_repository_root(start: Path) -> Path:
    start = start.expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "code").exists():
            return candidate
    return start


PROJECT_ROOT = Path(
    os.environ.get("GLIOMA_PROJECT_ROOT", find_repository_root(Path.cwd()))
).expanduser().resolve()

INPUT_H5AD = Path(
    os.environ.get(
        "GSE278450_POSTQC_H5AD",
        PROJECT_ROOT
        / "data"
        / "processed"
        / "DS4_GSE278450"
        / "GSE278450_postQC.h5ad",
    )
).expanduser().resolve()

OUTPUT_H5AD = Path(
    os.environ.get(
        "GSE278450_SCRUBLET_H5AD",
        PROJECT_ROOT
        / "data"
        / "processed"
        / "DS4_GSE278450"
        / "GSE278450_postQC_scrublet_checked.h5ad",
    )
).expanduser().resolve()

AUDIT_DIR = (
    PROJECT_ROOT
    / "results"
    / "qc"
    / "DS4_GSE278450"
    / "00c_doublet_diagnostic"
)
FIGURE_DIR = AUDIT_DIR / "figures"

EXPECTED_INPUT_SHAPE = (88_094, 24_584)
EXPECTED_PREDICTED_DOUBLETS = 310
EXPECTED_PREDICTED_SINGLETS = 87_784

REQUESTED_SCRUBLET_PARAMETERS = {
    "sim_doublet_ratio": 2.0,
    "expected_doublet_rate": 0.05,
    "n_prin_comps": 30,
    "random_state": 0,
}

OUTPUT_H5AD.parent.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

if not INPUT_H5AD.exists():
    raise FileNotFoundError(
        f"Post-QC input not found: {INPUT_H5AD}\n"
        "Run 00b_ds4_quality_control.ipynb first or set "
        "GSE278450_POSTQC_H5AD."
    )

if importlib.util.find_spec("scrublet") is None:
    raise ImportError(
        "The 'scrublet' package is required by scanpy.pp.scrublet. "
        "Install the repository environment before running this notebook."
    )

print("Input:", INPUT_H5AD)
print("Output:", OUTPUT_H5AD)

In [ ]:
def matrix_values(matrix):
    return matrix.data if sp.issparse(matrix) else np.asarray(matrix).ravel()


def assert_raw_counts(matrix, label: str):
    values = matrix_values(matrix)
    if values.size == 0:
        raise ValueError(f"{label} contains no nonzero values.")
    if not np.isfinite(values).all():
        raise ValueError(f"{label} contains non-finite values.")
    if np.any(values < 0):
        raise ValueError(f"{label} contains negative values.")
    if not np.equal(values, np.floor(values)).all():
        raise ValueError(f"{label} is not an integer raw-count matrix.")

In [ ]:
adata = sc.read_h5ad(INPUT_H5AD)

if adata.shape != EXPECTED_INPUT_SHAPE:
    raise AssertionError(
        f"Expected input shape {EXPECTED_INPUT_SHAPE}, observed {adata.shape}"
    )
if "sample_id" not in adata.obs:
    raise KeyError("adata.obs['sample_id'] is required.")
if "counts" not in adata.layers:
    raise KeyError("The post-QC object is missing layers['counts'].")

assert_raw_counts(adata.X, "adata.X")
assert_raw_counts(adata.layers["counts"], 'adata.layers["counts"]')

for column in ("doublet_score", "predicted_doublet"):
    if column in adata.obs:
        del adata.obs[column]
adata.uns.pop("scrublet", None)

print(adata)

In [ ]:
signature = inspect.signature(sc.pp.scrublet)
supported_parameters = set(signature.parameters)

scrublet_parameters = {
    key: value
    for key, value in REQUESTED_SCRUBLET_PARAMETERS.items()
    if key in supported_parameters
}

omitted_parameters = sorted(
    set(REQUESTED_SCRUBLET_PARAMETERS) - set(scrublet_parameters)
)
if omitted_parameters:
    print(
        "Parameters not supported by this Scanpy version and omitted:",
        omitted_parameters,
    )

print("Scrublet parameters used:", scrublet_parameters)

sc.pp.scrublet(
    adata,
    **scrublet_parameters,
)

required_columns = {"doublet_score", "predicted_doublet"}
missing = required_columns - set(adata.obs.columns)
if missing:
    raise RuntimeError(
        f"Scrublet completed without creating required columns: {missing}"
    )

adata.obs["predicted_doublet"] = (
    adata.obs["predicted_doublet"].astype(bool)
)

n_doublets = int(adata.obs["predicted_doublet"].sum())
n_singlets = int((~adata.obs["predicted_doublet"]).sum())

print(adata.obs["predicted_doublet"].value_counts())
print(adata.obs["doublet_score"].describe())

if (
    n_doublets != EXPECTED_PREDICTED_DOUBLETS
    or n_singlets != EXPECTED_PREDICTED_SINGLETS
):
    print(
        "WARNING: Scrublet calls differ from the historical run. "
        "Review Scanpy/Scrublet versions before release. "
        f"Observed doublets={n_doublets}, singlets={n_singlets}."
    )

In [ ]:
per_sample_summary = (
    adata.obs.groupby("sample_id", observed=True)
    .agg(
        cells=("predicted_doublet", "size"),
        predicted_doublets=("predicted_doublet", "sum"),
        mean_doublet_score=("doublet_score", "mean"),
        median_doublet_score=("doublet_score", "median"),
        maximum_doublet_score=("doublet_score", "max"),
    )
    .reset_index()
)
per_sample_summary["predicted_doublet_fraction"] = (
    per_sample_summary["predicted_doublets"]
    / per_sample_summary["cells"]
)
per_sample_summary.to_csv(
    AUDIT_DIR / "GSE278450_scrublet_summary_by_sample.tsv",
    sep="\t",
    index=False,
)

adata.obs[
    ["sample_id", "doublet_score", "predicted_doublet"]
].to_csv(
    AUDIT_DIR / "GSE278450_scrublet_per_cell_scores.tsv.gz",
    sep="\t",
    compression="gzip",
)

per_sample_summary.sort_values(
    "predicted_doublet_fraction",
    ascending=False,
)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.hist(
    adata.obs["doublet_score"].to_numpy(),
    bins=80,
    edgecolor="black",
)
ax.set_xlabel("Scrublet doublet score")
ax.set_ylabel("Cells")
ax.set_title("GSE278450 post-QC Scrublet scores")
fig.tight_layout()
fig.savefig(
    FIGURE_DIR / "GSE278450_scrublet_score_histogram.png",
    dpi=300,
    bbox_inches="tight",
)
plt.close(fig)

In [ ]:
doublet_summary = {
    "dataset_id": "DS4_GSE278450",
    "geo_accession": "GSE278450",
    "scope": "complete post-QC DS4 object",
    "method": "scanpy.pp.scrublet",
    "parameters_requested": REQUESTED_SCRUBLET_PARAMETERS,
    "parameters_used": scrublet_parameters,
    "input_cells": int(adata.n_obs),
    "input_genes": int(adata.n_vars),
    "predicted_doublets": n_doublets,
    "predicted_singlets": n_singlets,
    "predicted_doublet_fraction": float(n_doublets / adata.n_obs),
    "doublets_removed_from_main_analysis": False,
    "main_analysis_cells": int(adata.n_obs),
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "output_h5ad": str(OUTPUT_H5AD),
}

adata.uns["doublet_detection_provenance"] = {
    "scope": "post_QC_dataset_level_diagnostic",
    "method": "scanpy.pp.scrublet",
    "doublets_removed_from_main_analysis": "false",
    "predicted_doublets": n_doublets,
    "predicted_singlets": n_singlets,
    "parameters_json": json.dumps(
        scrublet_parameters,
        sort_keys=True,
    ),
}

with open(
    AUDIT_DIR / "GSE278450_scrublet_summary.json",
    "w",
    encoding="utf-8",
) as handle:
    json.dump(doublet_summary, handle, indent=2)

print(doublet_summary)

In [ ]:
if adata.shape != EXPECTED_INPUT_SHAPE:
    raise AssertionError(
        "The doublet diagnostic unexpectedly changed the object shape: "
        f"{adata.shape}"
    )

temporary_output = OUTPUT_H5AD.with_suffix(".tmp.h5ad")
if temporary_output.exists():
    temporary_output.unlink()

adata.write_h5ad(temporary_output, compression="gzip")
temporary_output.replace(OUTPUT_H5AD)

with sc.read_h5ad(OUTPUT_H5AD, backed="r") as check:
    if check.shape != EXPECTED_INPUT_SHAPE:
        raise AssertionError(
            f"Saved-file validation failed: observed {check.shape}"
        )
    for column in ("doublet_score", "predicted_doublet"):
        if column not in check.obs:
            raise AssertionError(
                f"Saved output is missing adata.obs[{column!r}]."
            )

print("Saved and validated:", OUTPUT_H5AD)